# HiGAN+ Recognizer-Crop -- Compare 4 Experiments

Run this notebook **after** the four training notebooks have produced runs:

- `kaggle_train_baseline.ipynb` -> `runs/gan_iam_crop_baseline-*`
- `kaggle_train_left_half.ipynb` -> `runs/gan_iam_crop_left_half-*`
- `kaggle_train_left_3q.ipynb` -> `runs/gan_iam_crop_left_3q-*`
- `kaggle_train_char_aligned.ipynb` -> `runs/gan_iam_crop_char_aligned-*`

Each Kaggle training session writes to `/kaggle/working/.../runs/` and Kaggle persists those outputs as a per-notebook 'output' dataset.  To compare, attach those four output datasets as **Inputs** to this notebook (Add Data -> Your Datasets) and point `RUNS_ROOT` below at the directory that contains all four `runs/<prefix>-*` folders.

If you are running this locally instead, just leave the default and the script picks up `HiGAN+/runs/` from your clone.

## 1. Clone repo (only needed for `scripts/compare_runs.py`)

In [ ]:
%cd /kaggle/working
BRANCH = 'feat/recog-random-crop'
![ -d higanplus ] || git clone --depth 1 -b $BRANCH https://github.com/ks2n/higanplus.git
%cd /kaggle/working/higanplus
!git fetch origin $BRANCH --depth=1 && git checkout $BRANCH && git pull --ff-only origin $BRANCH


## 2. Pick the runs directory

Kaggle: change to `/kaggle/input/<dataset-slug>/runs` after attaching the four training-output datasets.  Local: leave the default.

In [ ]:
import os

# Default: the runs/ dir produced by a local clone training run.
RUNS_ROOT = '/kaggle/working/higanplus/HiGAN+/runs'

# Common Kaggle pattern when training-output datasets are attached:
#   RUNS_ROOT = '/kaggle/input/higanplus-recog-crop-runs/runs'
# Edit the line above to point at whichever directory contains the
# four 'gan_iam_crop_<variant>-*' folders.

assert os.path.isdir(RUNS_ROOT), f'RUNS_ROOT not found: {RUNS_ROOT}'
print('contents of', RUNS_ROOT, ':')
for entry in sorted(os.listdir(RUNS_ROOT)):
    print(' ', entry)


## 3. Run the comparison script

Produces:

- `compare_report.md` -- the same markdown table this cell prints to stdout (best / final / mean / epoch-of-best per metric per experiment).
- `compare.csv` -- one row per training step, one column per `<experiment>__<tag>`.
- `compare_plots/*.png` -- one chart per scalar tag, four lines per chart.

In [ ]:
import subprocess, sys, os
from pathlib import Path

OUT = Path('/kaggle/working/compare_out')
OUT.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, 'scripts/compare_runs.py',
    '--runs-root', RUNS_ROOT,
    '--output', str(OUT / 'compare_report.md'),
    '--csv', str(OUT / 'compare.csv'),
    '--png', str(OUT / 'compare_plots'),
]
print('running:', ' '.join(cmd))
subprocess.check_call(cmd, cwd='/kaggle/working/higanplus')


## 4. Render the markdown report inline

In [ ]:
from IPython.display import Markdown, display
report_path = '/kaggle/working/compare_out/compare_report.md'
with open(report_path) as f:
    display(Markdown(f.read()))


## 5. Plot images, side by side

In [ ]:
from IPython.display import Image, display
import glob, os
for png in sorted(glob.glob('/kaggle/working/compare_out/compare_plots/*.png')):
    print(os.path.basename(png))
    display(Image(png))


## 6. (Optional) Read the raw CSV with pandas

For your own analysis -- e.g. computing the mean of FID over the last 5 epochs, or building custom plots.

In [ ]:
import pandas as pd
df = pd.read_csv('/kaggle/working/compare_out/compare.csv')
print('rows:', len(df), '| columns:', len(df.columns))
df.head()
